# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is described by a [Croissant schema](https://mlcommons.org/croissant/) and available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print("\nDescription:")
print(metadata.description)

## 2. Data Overview
Review available record sets, their fields, and associated IDs. All entities are referenced by their `@id` fields as per the Croissant specification.

In [ ]:
# List all record sets and their fields with IDs
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"Record set: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for field in rs.fields:
                print(f"    Field: {field.name}, @id: {field.id}, Type: {field.data_type if hasattr(field, 'data_type') else 'n/a'}")
else:
    print('No record sets defined in this dataset. Please refer to the documentation or metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** In this dataset, each data table or set is identified by its own unique `@id`. Adjust the target record set accordingly.

In [ ]:
# Find available record sets
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    # Collect @ids
    record_set_ids = [rs.id for rs in metadata.record_sets]
    print("Available record sets by @id:")
    for rid in record_set_ids:
        print("  ", rid)
else:
    record_set_ids = []
    print('No record sets available for extraction.')

# Attempt to load dataframes from each record set (if any)
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for record set {record_set_id}")

# Show columns of the first available record set
if record_set_ids:
    first_record_set = record_set_ids[0]
    print(f"\nFirst record set columns (@id): {list(dataframes[first_record_set].columns)}")
    display(dataframes[first_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include handling missing values and basic grouping for summary statistics.

**Note:** Replace the placeholders below (`<numeric_field_id>`, `<group_field_id>`) with actual field `@id`s from the record set you wish to analyze. The cell will show some example logic for one numeric field (if available).

In [ ]:
# Identify a record set and numeric field for EDA (update these @ids as needed from above)

if record_set_ids:
    record_set_id = record_set_ids[0]  # Use the first record set
    df = dataframes[record_set_id].copy()
    # Try to identify a numeric field by data type
    fields = []
    for rs in metadata.record_sets:
        if rs.id == record_set_id:
            fields = rs.fields if hasattr(rs, 'fields') else []
            break
    numeric_field_id = None
    for field in fields:
        # The Croissant data type may be "Float", "Integer", "Number"
        if hasattr(field, 'data_type') and str(field.data_type).lower() in ['number', 'float', 'integer']:
            numeric_field_id = field.id
            print(f"Using numeric field: {field.name} (@id: {field.id})")
            break
    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Drop missing
        filtered_df = df[df[numeric_field_id].notna() & (df[numeric_field_id] > 0)]
        print(f"Filtered records with {numeric_field_id} > 0: {len(filtered_df)} rows")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by another field (categorical)
        group_field_id = None
        for field in fields:
            if hasattr(field, 'data_type') and str(field.data_type).lower() not in ['number','float','integer']:
                if field.id in filtered_df.columns:
                    group_field_id = field.id
                    print(f"Grouping by field: {field.name} (@id: {field.id})")
                    break
        if group_field_id:
            # Use numeric field (original, unnormalized)
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print('No numeric field detected or available in this record set.')
else:
    print('No record set data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example histogram and bar plot.

Replace field `@id`s as needed based on your exploration above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting only if numeric field was detected
if record_set_ids and 'numeric_field_id' in locals() and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Bar plot by group field
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        plot_df = df.groupby(group_field_id)[numeric_field_id].mean().sort_values().reset_index()
        sns.barplot(x=group_field_id, y=numeric_field_id, data=plot_df)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded the FAIR² dataset Croissant schema using the `mlcroissant` library.
- Explored record sets and fields by their unique `@id` references.
- Extracted tabular data into DataFrames and demonstrated basic filtering and normalization on numeric fields.
- Provided examples for exploratory visualizations.

The approach ensures robust, future-proof data handling using Croissant entity identifiers. For further analyses or machine learning tasks, build on this workflow by referencing `@id` fields for all data manipulation.